In [ ]:
%load_ext autoreload
%autoreload 2


import matplotlib.pyplot as plt
from mob_mod.utils import LOGGER
from mob_mod import datasets
from mob_mod.config import PATHS
from mob_mod.plot.setup import cm

## post stratified 

In [ ]:
gdf_lsoa_london_2021 = datasets.load_lsoa_2021(city="London", version="BFC")
# ddf_mobility_2021 = datasets.load_mobility_2021()
dff_user_home = datasets.load_user_home()
df_census_london = datasets.load_lsoa_census_london(rerun=False)

### 计算每个lsoa的权重
1. 计算每个lsoa的pop2021 和 用户数
2. 计算每个lsoa的pop_scalerate
3. 合并到df_mob_home 得到每个用户的pop_scalerate

In [ ]:
df_user_lsoacount = (
    dff_user_home.groupby("lsoa21cd")[["userid"]]
    .count()
    .reset_index()
    .compute()
    .rename(columns={"userid": "usercount"})
)


def get_lsoa_prpty(df_census_london, df_user_lsoacount):
    df_lsoa_prpty = df_census_london.merge(df_user_lsoacount, on="lsoa21cd", how="left")
    overall_rate = (
        df_lsoa_prpty["pop_lsoa_2021"].sum() / df_lsoa_prpty["usercount"].sum()
    )
    LOGGER.info(f"overall pop:user rate  = {overall_rate}")
    df_lsoa_prpty = df_lsoa_prpty.assign(
        pop_scalerate=df_lsoa_prpty["pop_lsoa_2021"]
        / (df_lsoa_prpty["usercount"] * overall_rate)
    )
    # df_lsoa_prpty.pop_scalerate.hist(bins=50)
    return df_lsoa_prpty.sort_values("lsoa21cd").reset_index(drop=True)


df_lsoa_prpty = get_lsoa_prpty(df_census_london, df_user_lsoacount)
dff_user_prpty = dff_user_home.merge(df_lsoa_prpty, on="lsoa21cd", how="left").compute()

In [ ]:
all(df_lsoa_prpty["lsoa21cd"] == gdf_lsoa_london_2021["lsoa21cd"])

- 房价更低，人口密度更低，教育水平更低，失业率更低？

In [ ]:
df_lsoa_prpty[df_lsoa_prpty.isna().any(axis=1)]
# 均值填充了mean_houseprice_lsoa_2021，income_msoa_2021后，有2个lsoa没有user

In [ ]:
df_lsoa_prpty[
    [
        "pop_lsoa_2021",
        "mean_houseprice_lsoa_2021",
        "popdens_lsoa_2021",
        "edu_lsoa_2021",
        "unemployed_lsoa_2021",
        "income_msoa_2021",
    ]
].corr()
# 确保变量之间的相关关系是正常的

In [ ]:
plot_dict = {
    "pop_lsoa_2021_decile": {
        "title": "Population",
    },
    "popdens_lsoa_2021_decile": {
        "title": "Population Density",
    },
    "mean_houseprice_lsoa_2021_decile": {
        "title": "Mean price paid for residential properties",
    },
    "edu_lsoa_2021_decile": {
        "title": "Share of Level 4 qualifications or above",
    },
    "unemployed_lsoa_2021_decile": {
        "title": "Share of Unemployed",
    },
    "income_msoa_2021_decile": {
        "title": "Total annual income",
    },
}

fig, ax = plt.subplots(2, 3, figsize=(18.3 * cm, 10 * cm))
ax = ax.flatten()
for i, col in enumerate(plot_dict.keys()):
    deciles_before = dff_user_prpty.groupby(col)["userid"].count()
    deciles_after = dff_user_prpty.groupby(col)["pop_scalerate"].sum("pop_scalerate")
    ax[i].scatter(
        range(1, 11),
        deciles_before / deciles_before.sum(),
        marker="o",
        s=8,
        zorder=10,
        label="Original",
    )
    ax[i].scatter(
        range(1, 11),
        deciles_after / deciles_after.sum(),
        marker="o",
        s=8,
        zorder=10,
        label="Post-Stratified",
    )
    ax[i].set_title(plot_dict[col]["title"])
    ax[i].hlines(0.1, 0, 11, linestyles="dashed", color="grey")
    ax[i].set_ylim(0.06, 0.14)
    ax[i].set_xlim(0, 11)
    ax[i].set_xlabel("Decile")
    ax[i].set_ylabel("Proportion")
ax[0].legend()
fig.suptitle("Sampling Distribution Across LSOA Deciles", fontsize=10)
plt.tight_layout()
plt.show()

### 计算od
1. 根据oa筛选城市内的od
2. 与userid合并，得到每个userid的权重赋予给od
3. 计算lsoa之间的加权od

In [ ]:
ddf_user_mob = datasets.load_mobility_2021()

# 伦敦的oa
oa_london = datasets.load_oa_lookup(level="oa")
oa_london = oa_london[oa_london.uagc16nm == "London"][["oa21cd", "lsoa21cd"]]

In [ ]:
# 筛选出住在伦敦的人的，在伦敦范围内OA层级的OD，并且OD的起点和终点不相同
ddf_od_oa = ddf_user_mob[
    (ddf_user_mob["label"] == -1)
    & ddf_user_mob.userid.isin(dff_user_prpty.userid)
    & (ddf_user_mob.o_oa != ddf_user_mob.d_oa)
    & ddf_user_mob.o_oa.isin(oa_london.oa21cd)
    & ddf_user_mob.d_oa.isin(oa_london.oa21cd)
]

# 给每个流一个权重 和类别
ddf_od_w = ddf_od_oa.merge(
    dff_user_prpty[["userid", "mean_houseprice_lsoa_2021_decile", "pop_scalerate"]],
    on="userid",
)

# 合并到lsoa
ddf_od_w = ddf_od_w.merge(
    oa_london, left_on="o_oa", right_on="oa21cd", how="left"
).merge(oa_london, left_on="d_oa", right_on="oa21cd", how="left", suffixes=("_o", "_d"))

# 计算lsoa之间的加权od
df_odsize = (
    ddf_od_w[["lsoa21cd_o", "lsoa21cd_d", "pop_scalerate"]]
    .groupby(["lsoa21cd_o", "lsoa21cd_d"])
    .sum("pop_scalerate")
    .compute()
    .reset_index()
    .rename(columns={"pop_scalerate": "od_size"})
)

# LSOAid 映射
df_lsoa_prpty["id"] = df_lsoa_prpty.index

df_odsize["lsoa_o_id"] = df_odsize.lsoa21cd_o.map(
    df_lsoa_prpty.set_index("lsoa21cd")["id"]
)
df_odsize["lsoa_d_id"] = df_odsize.lsoa21cd_d.map(
    df_lsoa_prpty.set_index("lsoa21cd")["id"]
)
df_odsize

##### Optional

In [ ]:
# odsize sum
LOGGER.info(f"odsize sum = {df_odsize.od_size.sum()}")  # 5477790.139074816

# 计算self loops 的od size
LOGGER.info(
    f"self loops odsize sum = {df_odsize[df_odsize.lsoa21cd_o == df_odsize.lsoa21cd_d].od_size.sum()}"
)

In [ ]:
## how many LSOAs in London have no mobility data?
set_lsoa_hasmob = set(df_odsize.lsoa21cd_o.unique()).union(
    set(df_odsize.lsoa21cd_d.unique())
)
LOGGER.info(
    f"LSOA21CD with no mob data: {len(set(oa_london.lsoa21cd).difference(set_lsoa_hasmob))}"
)

## export

In [ ]:
df_lsoa_prpty.to_parquet(PATHS["processed"] / "lsoa_prpty.parquet")

- 保留两位小数

In [ ]:
# remove self loops
df_odsize_no_self = df_odsize[df_odsize["lsoa21cd_o"] != df_odsize["lsoa21cd_d"]]
df_odsize_no_self.to_parquet(PATHS["processed"] / "odsize_no_self.parquet")

- 来自房价更低区域的人的出行更频繁？
- 人们更喜欢在同一个类型的地点之间出行？